# SPF vs. simple forecasting models

Compare the SPF panel against a few simple time-series models on the same indicator. Models are evaluated in **real time**: at survey quarter `S` we use only the RTDSM vintage published in `S`, restricted to data through `S-1` — exactly what an SPF forecaster would have seen.

Pipeline:
1. Load SPF microdata and RTDSM for the indicator.
2. Compute the SPF squared/absolute-error panel.
3. Generate model forecasts at every target quarter, with the same vintage discipline.
4. Concatenate model error columns with the top-N SPF forecasters.
5. Run the marginal rank CI procedure on the joint panel — models become extra "forecasters" in the ranking.

In [1]:
import numpy as np
import pandas as pd
from functools import partial

from rankci import (
    load_spf,
    load_rtdsm,
    compute_error_panel,
    select_top_forecasters,
    model_error_panel,
    forecast_naive,
    forecast_ar1,
    rank_ci_marginal_pairwise,
    rank_ci_stepwise_pairwise,
)

# Configuration

Set the indicator and parameters here. Switch indicator/RTDSM and the rest of the notebook follows.

In [7]:
# --- Indicator ---
INDICATOR     = "NGDP"
RTDSM_PATH    = "../data/NOUTPUTQvQd.xlsx"
RTDSM_PREFIX  = "NOUTPUT"
RTDSM_FREQ    = "quarterly"

# Alternative: unemployment
# INDICATOR    = "UNEMP"
# RTDSM_PATH   = "../data/rucQvMd.xlsx"
# RTDSM_PREFIX = "RUC"
# RTDSM_FREQ   = "monthly"

# --- Comparison knobs ---
HORIZON   = 3              # SPF horizon (3 = one-quarter-ahead)
METRIC    = "squared"      # "squared" → MSE, "absolute" → MAE
N_SPF     = 8              # how many SPF forecasters to keep
ALPHA     = 0.2           # 1-alpha confidence
B         = 5000           # bootstrap replications
SEED      = 42

# Load data and build the SPF panel

In [3]:
df    = load_spf("../data/SPFmicrodata.xlsx", sheet=INDICATOR)
rtdsm = load_rtdsm(RTDSM_PATH, prefix=RTDSM_PREFIX, freq=RTDSM_FREQ)

X_spf_full = compute_error_panel(df, rtdsm, indicator=INDICATOR, horizon=HORIZON, metric=METRIC)
X_spf_top  = select_top_forecasters(X_spf_full, N=N_SPF, min_obs=20)

print(f"SPF panel: {X_spf_full.shape[0]} target quarters × {X_spf_full.shape[1]} forecasters total")
print(f"Top-{N_SPF} subset: {X_spf_top.shape[0]} target quarters × {X_spf_top.shape[1]} forecasters")
print(f"Forecaster IDs: {X_spf_top.columns.tolist()}")
print(f"Sample range: {X_spf_top.index[0]} → {X_spf_top.index[-1]}")

/Users/Parimah/anaconda3/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


SPF panel: 225 target quarters × 345 forecasters total
Top-8 subset: 225 target quarters × 8 forecasters
Forecaster IDs: [65, 426, 84, 433, 428, 411, 421, 40]
Sample range: (1968, 4) → (2025, 1)


# Define and run the models

Add or remove models here. Each entry is `name → callable(history, h)`. Use `partial` to bind extra kwargs (e.g. AR(1) `transform`).

In [4]:
models = {
    "naive":         forecast_naive,
    "ar1_levels":    partial(forecast_ar1, transform="levels"),
    "ar1_log_diff":  partial(forecast_ar1, transform="log_diff"),
}

target_quarters = list(X_spf_top.index)

X_models = model_error_panel(
    rtdsm,
    models,
    target_quarters,
    spf_horizon=HORIZON,
    metric=METRIC,
)

# Drop quarters where any model returned NaN (typically early-sample warmup)
X_models = X_models.dropna(how="any")

print(f"Model panel: {X_models.shape[0]} quarters × {X_models.shape[1]} models")
print(f"\nModel mean {METRIC} error:")
print(X_models.mean().round(2))

Model panel: 224 quarters × 3 models

Model mean squared error:
naive           183451.15
ar1_levels       87872.84
ar1_log_diff    122080.54
dtype: float64


# Concatenate and rank

The models become extra "forecasters" in the panel and are ranked alongside the SPF top-N.

In [5]:
# Align both panels to the intersection of target quarters
common_idx = X_spf_top.index.intersection(X_models.index)
X_full = pd.concat(
    [X_spf_top.loc[common_idx], X_models.loc[common_idx]],
    axis=1,
)
print(f"Joint panel: {X_full.shape[0]} quarters × {X_full.shape[1]} (forecasters + models)")
print(f"Columns: {X_full.columns.tolist()}")

Joint panel: 224 quarters × 11 (forecasters + models)
Columns: [65, 426, 84, 433, 428, 411, 421, 40, 'naive', 'ar1_levels', 'ar1_log_diff']


In [8]:
X = X_full.values
ids = [str(c) for c in X_full.columns]

out = rank_ci_marginal_pairwise(X, alpha=ALPHA, B=B, seed=SEED)

metric_col = "MSE" if METRIC == "squared" else "MAE"

results = pd.DataFrame({
    "ID":       ids,
    "kind":     ["model" if c in X_models.columns else "SPF" for c in X_full.columns],
    metric_col: out["theta_hat"].round(2),
    "cv_j":     out["critical_values"].round(3),
    "CI_lower": out["rank_ci"][:, 0],
    "CI_upper": out["rank_ci"][:, 1],
}).sort_values(metric_col)
results.index = range(1, len(results) + 1)
results.index.name = "Rank"
results

,ID,kind,MSE,cv_j,CI_lower,CI_upper
Rank,,,,,,
1,40,SPF,7356.44,2.509,1,10
2,65,SPF,8430.35,1.957,3,10
3,84,SPF,9777.84,2.509,1,10
4,ar1_levels,model,87872.84,3.420,1,11
5,428,SPF,106804.66,3.265,1,11
6,421,SPF,107271.95,3.032,1,11
7,ar1_log_diff,model,122080.54,3.026,1,11
8,426,SPF,122870.13,2.774,1,10
9,411,SPF,134677.93,2.715,1,11
